# Bootstrap Validation — Simple Fixed-Param Models

Tests whether the pipeline's AUC comes from the signal in the data or from the tuning machinery.

For each of **B=100 bootstrap resamples**, three models are fit with **fixed, depth-restricted hyperparameters** on **all 26 features** (no Optuna, no SHAP, no feature selection). Test AUC is evaluated on the fixed held-out test set.

**What this answers:** If AUC distributions here are similar to nb26 (full pipeline, Optuna 10 trials), the signal is in the data and the tuning machinery adds little. If noticeably lower, the pipeline earns its complexity.

**Simplifications vs nb21/nb26:**
- No Optuna — fixed reasonable params with depth restriction
- No feature selection — all 26 features used
- No genre consolidation, no centrality ablation

**Models:** CatBoost · XGBoost · Random Forest · Stratified Baseline  
Results saved to `bootstrap_simple_results.pkl` after every iteration.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import matplotlib.gridspec as gridspec

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import train_test_split, StratifiedKFold

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [2]:
B            = 100
RANDOM_STATE = 42
BOOT_MODELS  = ['CatBoost', 'XGBoost', 'Random Forest']

# nb21 single-run test AUC — reference lines on histograms
NB21_AUC = {
    'CatBoost':      0.7533,
    'XGBoost':       0.7742,
    'Random Forest': 0.7671,
}

# Fixed params — depth-restricted, no tuning
FIXED_PARAMS = {
    'Random Forest': {
        'n_estimators': 200,
        'max_depth':    5,
        'min_samples_leaf': 5,
        'max_features': 'sqrt',
        'class_weight': 'balanced',
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
    },
    'XGBoost': {
        'n_estimators':  200,
        'learning_rate': 0.05,
        'max_depth':     3,
        'subsample':     0.8,
        'colsample_bytree': 0.8,
        'random_state':  RANDOM_STATE,
        'eval_metric':   'logloss',
        'verbosity':     0,
    },
    'CatBoost': {
        'iterations':    200,
        'learning_rate': 0.05,
        'depth':         4,
        'random_seed':   RANDOM_STATE,
        'verbose':       0,
    },
}

df = pd.read_csv('../df_artists_final.csv', index_col=0).reset_index()
X  = df.drop(columns=['top_20_hitmaker'])
y  = df['top_20_hitmaker']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Class balance (train): {y_train.mean():.3f} hitmaker')

Train: (607, 26)  Test: (152, 26)
Class balance (train): 0.432 hitmaker


In [3]:
def build_model(name):
    p = FIXED_PARAMS[name]
    if name == 'Random Forest':
        return RandomForestClassifier(**p)
    elif name == 'XGBoost':
        return XGBClassifier(**p)
    elif name == 'CatBoost':
        return CatBoostClassifier(**p)
    raise ValueError(f'Unknown model: {name}')


def run_one_model(name, X_tr, y_tr, X_te, y_te):
    imp      = SimpleImputer(strategy='median')
    X_tr_imp = pd.DataFrame(imp.fit_transform(X_tr), columns=X_tr.columns)
    X_te_imp = pd.DataFrame(imp.transform(X_te),     columns=X_te.columns)

    model = build_model(name)
    model.fit(X_tr_imp, y_tr)

    train_proba = model.predict_proba(X_tr_imp)[:, 1]
    test_proba  = model.predict_proba(X_te_imp)[:, 1]

    train_auc = roc_auc_score(y_tr, train_proba)
    test_auc  = roc_auc_score(y_te, test_proba)

    return {
        'test_auc':  test_auc,
        'train_auc': train_auc,
        'gap':       train_auc - test_auc,
        'brier':     brier_score_loss(y_te, test_proba),
    }

In [ ]:
imp = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imp.fit_transform(X_train), columns=X_train.columns)
X_test_imp  = pd.DataFrame(imp.transform(X_test),      columns=X_test.columns)

print(f'{"Model":<20}  {"Train AUC":>10}  {"Test AUC":>10}  {"Gap":>8}')
print('─' * 55)

for name in BOOT_MODELS:
    model = build_model(name)
    model.fit(X_train_imp, y_train)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train_imp)[:, 1])
    test_auc  = roc_auc_score(y_test,  model.predict_proba(X_test_imp)[:, 1])
    gap       = train_auc - test_auc
    print(f'{name:<20}  {train_auc:>10.4f}  {test_auc:>10.4f}  {gap:>8.4f}')

## Bootstrap loop

B=100 independent iterations. Saved to `bootstrap_simple_results.pkl` after every iteration.

In [4]:
results = {name: {'test_auc': [], 'train_auc': [], 'gap': [], 'brier': []} for name in BOOT_MODELS}
results['Baseline'] = {'test_auc': [], 'brier': []}

rng = np.random.default_rng(RANDOM_STATE)

for b in range(B):
    seed     = RANDOM_STATE + b
    boot_idx = rng.choice(len(X_train), size=len(X_train), replace=True)
    X_tr_b   = X_train.iloc[boot_idx].reset_index(drop=True)
    y_tr_b   = y_train.iloc[boot_idx].reset_index(drop=True)

    print(f'[{b+1:03d}/{B}] ', end='', flush=True)

    for name in BOOT_MODELS:
        print(f'{name}... ', end='', flush=True)
        try:
            res = run_one_model(name, X_tr_b, y_tr_b, X_test, y_test)
            results[name]['test_auc'].append(res['test_auc'])
            results[name]['train_auc'].append(res['train_auc'])
            results[name]['gap'].append(res['gap'])
            results[name]['brier'].append(res['brier'])
            print(f'AUC={res["test_auc"]:.3f} gap={res["gap"]:.3f} | ', end='', flush=True)
        except Exception as e:
            print(f'ERROR({e}) | ', end='', flush=True)
            for k in ('test_auc', 'train_auc', 'gap', 'brier'):
                results[name][k].append(np.nan)

    dummy = DummyClassifier(strategy='stratified', random_state=seed)
    dummy.fit(X_tr_b, y_tr_b)
    dummy_proba = dummy.predict_proba(X_test)[:, 1]
    results['Baseline']['test_auc'].append(roc_auc_score(y_test, dummy_proba))
    results['Baseline']['brier'].append(brier_score_loss(y_test, dummy_proba))
    print(f'Baseline={results["Baseline"]["test_auc"][-1]:.3f}')

    with open('bootstrap_simple_results.pkl', 'wb') as f:
        pickle.dump(results, f)

print('\nDone. Results saved to bootstrap_simple_results.pkl')

[001/100] CatBoost... AUC=0.747 gap=0.211 | XGBoost... AUC=0.718 gap=0.234 | Random Forest... AUC=0.732 gap=0.165 | Baseline=0.492
[002/100] CatBoost... AUC=0.784 gap=0.193 | XGBoost... AUC=0.749 gap=0.218 | Random Forest... AUC=0.769 gap=0.137 | Baseline=0.506
[003/100] CatBoost... AUC=0.798 gap=0.174 | XGBoost... AUC=0.782 gap=0.181 | Random Forest... AUC=0.783 gap=0.131 | Baseline=0.501
[004/100] CatBoost... AUC=0.767 gap=0.206 | XGBoost... AUC=0.742 gap=0.219 | Random Forest... AUC=0.772 gap=0.126 | Baseline=0.511
[005/100] CatBoost... AUC=0.741 gap=0.231 | XGBoost... AUC=0.723 gap=0.241 | Random Forest... AUC=0.736 gap=0.162 | Baseline=0.502
[006/100] CatBoost... AUC=0.766 gap=0.203 | XGBoost... AUC=0.753 gap=0.211 | Random Forest... AUC=0.764 gap=0.130 | Baseline=0.522
[007/100] CatBoost... AUC=0.760 gap=0.220 | XGBoost... AUC=0.766 gap=0.204 | Random Forest... AUC=0.769 gap=0.145 | Baseline=0.463
[008/100] CatBoost... AUC=0.765 gap=0.208 | XGBoost... AUC=0.747 gap=0.215 | Random

## Results

**Reading the histograms:**
- Dashed line = bootstrap mean · Shaded band = 90% CI · Dotted line = nb21 single-run AUC
- Simple mean ≈ nb21 tuned mean → tuning adds little; signal is in the data
- Simple mean notably lower → pipeline tuning is earning its complexity
- All model distributions clearly above Baseline → real signal, not noise

In [ ]:
with open('bootstrap_simple_results.pkl', 'rb') as f:
    results = pickle.load(f)

# nb26 means for comparison
NB26_MEAN = {
    'CatBoost':      0.7187,
    'XGBoost':       0.7391,
    'Random Forest': 0.7447,
}

all_names = BOOT_MODELS + ['Baseline']
palette = {
    'CatBoost':      '#4C72B0',
    'XGBoost':       '#DD8452',
    'Random Forest': '#55A868',
    'Baseline':      '#aaaaaa',
}

xlim_auc = (0.40, 0.90)
x_auc    = np.linspace(*xlim_auc, 300)

baseline_aucs = [v for v in results['Baseline']['test_auc'] if not np.isnan(v)]
baseline_mean = np.mean(baseline_aucs)

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.55, wspace=0.30)

top_axes    = [fig.add_subplot(gs[0, col]) for col in range(4)]
ax_combined = fig.add_subplot(gs[1, :])

fig.suptitle(
    f'Bootstrap Validation — Simple Fixed-Param Models  (B={B})\n'
    'No Optuna, no feature selection — depth-restricted defaults on all 26 features.',
    fontsize=13, fontweight='bold')

# ── Row 1: Test AUC histograms ──
for col, name in enumerate(all_names):
    ax   = top_axes[col]
    vals = [v for v in results[name]['test_auc'] if not np.isnan(v)]
    if not vals:
        ax.set_title(f'{name}\n(no data)'); continue

    color = palette[name]
    mu    = np.mean(vals)
    p5    = np.percentile(vals, 5)
    p95   = np.percentile(vals, 95)

    ax.hist(vals, bins=15, color=color, alpha=0.35, density=True)
    kde   = gaussian_kde(vals, bw_method='scott')
    y_kde = kde(x_auc)
    ax.plot(x_auc, y_kde, color=color, lw=2.0)

    mask = (x_auc >= p5) & (x_auc <= p95)
    ax.fill_between(x_auc[mask], y_kde[mask], alpha=0.30, color=color,
                    label=f'90% CI [{p5:.3f}, {p95:.3f}]')
    ax.axvline(mu, color=color, lw=2.0, linestyle='--', label=f'mean={mu:.3f}')
    if name in NB21_AUC:
        ax.axvline(NB21_AUC[name], color='black', lw=1.5, linestyle=':',
                   label=f'nb21={NB21_AUC[name]:.3f}')
    if name in NB26_MEAN:
        ax.axvline(NB26_MEAN[name], color='grey', lw=1.5, linestyle='--',
                   label=f'nb26={NB26_MEAN[name]:.3f}')

    if name in BOOT_MODELS:
        gap_vs_base = mu - baseline_mean
        ax.text(0.97, 0.95, f'{gap_vs_base:+.3f} vs baseline',
                transform=ax.transAxes, ha='right', va='top', fontsize=8,
                color=color, bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                       edgecolor=color, alpha=0.85))

    ax.set_xlabel('Test AUC', fontsize=9)
    ax.set_ylabel('Density' if col == 0 else '', fontsize=9)
    ax.set_title(f'{name}\nTest AUC', fontweight='bold')
    ax.set_xlim(xlim_auc)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

# ── Row 2: Combined AUC overlay ──
for name in BOOT_MODELS:
    vals = [v for v in results[name]['test_auc'] if not np.isnan(v)]
    if not vals: continue
    color = palette[name]
    mu    = np.mean(vals)
    kde   = gaussian_kde(vals, bw_method='scott')
    y_kde = kde(x_auc)
    ax_combined.plot(x_auc, y_kde, color=color, lw=2.5,
                     label=f'{name}  (mean={mu:.3f})')
    ax_combined.fill_between(x_auc, y_kde, alpha=0.12, color=color)
    ax_combined.axvline(mu, color=color, lw=1.5, linestyle='--', alpha=0.7)

ax_combined.axvline(baseline_mean, color='#888888', lw=1.5, linestyle=':',
                    label=f'Baseline mean={baseline_mean:.3f}')
ax_combined.set_xlabel('Test AUC', fontsize=10)
ax_combined.set_ylabel('Density', fontsize=10)
ax_combined.set_title('All Models — Combined Test AUC Distributions', fontweight='bold')
ax_combined.set_xlim(xlim_auc)
ax_combined.legend(fontsize=9)
ax_combined.grid(True, alpha=0.3)

plt.savefig('bootstrap_simple_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(f'{"Model":<20}  {"AUC mean":>8}  {"AUC std":>8}  {"90% CI":>22}  {"Brier":>7}  {"B":>5}')
print('─' * 80)

for name in all_names:
    aucs   = [v for v in results[name]['test_auc'] if not np.isnan(v)]
    briers = [v for v in results[name]['brier']    if not np.isnan(v)]
    if not aucs: continue
    ci = f'[{np.percentile(aucs, 5):.3f}, {np.percentile(aucs, 95):.3f}]'
    print(f'{name:<20}  {np.mean(aucs):>8.4f}  {np.std(aucs):>8.4f}  {ci:>22}  {np.mean(briers):>7.4f}  {len(aucs):>5d}')

print()
print('Comparison vs nb26 (full pipeline, Optuna 10 trials):')
for name in BOOT_MODELS:
    aucs = [v for v in results[name]['test_auc'] if not np.isnan(v)]
    if aucs and name in NB26_MEAN:
        delta = np.mean(aucs) - NB26_MEAN[name]
        note  = '  tuning adds little' if abs(delta) < 0.01 else (
                '  simple wins (!)' if delta > 0.01 else '  tuning helps')
        print(f'  {name:<20}  simple={np.mean(aucs):.4f}  nb26={NB26_MEAN[name]:.4f}  delta={delta:+.4f}{note}')